# 07 · Decoder, Loss, and Generation

**Goal:** see how the decoder differs from the encoder, how the output scores (*logits*) become a loss during training, and how tokens are generated one at a time during inference.

Concepts: *masked self-attention*, *cross-attention*, *logits*, *softmax over vocabulary*, *cross-entropy*, *perplexity*, *autoregressive / greedy decoding*, *temperature*.

Standalone — needs `numpy` and `matplotlib`.

## Words you'll meet
- **Logits**: the raw scores the model gives every possible next word, before they are turned into percentages. Like judges' raw points before you work out who won.
- **Softmax**: a recipe that turns raw scores into percentages that are all positive and add up to 100%.
- **Cross-entropy (loss)**: a penalty for being wrong, bigger when the model was confidently wrong. Like losing more marks for a confident wrong answer than for an unsure one.
- **Perplexity**: "how many options does the model feel torn between?" Lower means more sure of itself.
- **Masked / causal attention**: a word may only look at words before it, like covering the rest of a sentence with your hand while you guess the next word.
- **Cross-attention**: the decoder (the writer) glancing back at what the encoder (the reader) understood about the input, like checking your notes while writing an answer.
- **Autoregressive**: writing a sentence one word at a time, each new word based on everything you have written so far.
- **Greedy**: always pick the favourite. **Temperature**: how adventurous you are when choosing; low is cautious, high is wild.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

> 🧒 **In plain words:**
> This cell loads two tools: `numpy` (fast maths on lists of numbers) and `matplotlib` (for drawing graphs).
> It also defines `softmax`, a recipe that turns any list of scores into percentages that add up to 1.
> Line by line: subtracting the biggest score first stops numbers getting huge (it does not change the answer); `np.exp` makes every score positive; dividing by the total makes them add up to 1.
> Nothing prints here, it just sets things up.

## 1. Decoder layer = encoder layer + two changes
```
decoder input ─► MASKED self-attn ─► Add&Norm ─► CROSS-attn (K,V from encoder) ─► Add&Norm ─► FFN ─► Add&Norm
```
1. The first attention is **causal** (a token can't see the future).
2. A middle **cross-attention** sub-layer: queries come from the decoder, keys/values from the encoder output ("memory").

Shapes with `T_enc = 5` source tokens and `T_dec = 3` target tokens:

> 🧒 **Think of it like...** a student writing an essay answer. *Masked self-attention* is covering the rest of the page with a ruler so you can only reread what you have already written. *Cross-attention* is glancing up at your source notes (the encoder's output) to decide what to write next.

In [ ]:
rng = np.random.default_rng(0)
B, T_enc, T_dec, d = 1, 5, 3, 8

memory = rng.normal(size=(B, T_enc, d))          # encoder output
y = rng.normal(size=(B, T_dec, d))               # decoder states

# masked self-attention weights: (T_dec, T_dec), lower-triangular
s = y @ y.transpose(0, 2, 1) / np.sqrt(d)
s = np.where(np.tril(np.ones((T_dec, T_dec), bool)), s, -np.inf)
w_self = softmax(s)

# cross-attention weights: (T_dec, T_enc)
c = y @ memory.transpose(0, 2, 1) / np.sqrt(d)
w_cross = softmax(c)

print("self  attention weights", w_self.shape, "\n", w_self[0].round(2))
print("cross attention weights", w_cross.shape, "= (T_dec, T_enc)\n", w_cross[0].round(2))

> 🧒 **In plain words:**
> We invent a tiny example: 5 "source" words that the encoder understood (`memory`) and 3 "target" words the decoder is writing (`y`), each described by 8 numbers.
> `y @ y.transpose(...)` compares every target word with every other target word (a similarity score). `np.tril` makes a staircase shape of True/False, and `np.where` replaces every "future" score with minus infinity, so after softmax those get exactly 0% attention.
> The cross-attention does the same comparison, but between target words and source words.
> **Notice:** in the self-attention table the numbers above the diagonal are 0, so word 1 can only look at itself (it gets 1.0), word 2 at words 1 and 2, word 3 at all three. In the cross table every row has 5 numbers (one per source word) and sums to 1. For example row 1 mostly looks at source word 5 (0.54), and row 2 at source word 4 (0.59).
> **Try it:** change `default_rng(0)` to `default_rng(1)` and see the attention pattern change, but the zeros in the self-attention table stay.

Each *target* position distributes attention over all *source* positions (rows sum to 1) — e.g. when emitting the 3rd sorted digit, look at the right input digit.

## 2. From vectors to a distribution over the vocabulary
The decoder's final output `(B, T, d_model)` goes through a linear layer to **`V` scores per position** — **logits** `(B, T, V)`. Softmax turns them into probabilities of "which token comes next".

In [ ]:
V = 13
B, T = 2, 4
decoder_out = rng.normal(size=(B, T, d))
W_out = rng.normal(size=(d, V)) / np.sqrt(d)

logits = decoder_out @ W_out
probs = softmax(logits, axis=-1)
print("logits", logits.shape, "= (B, T, V)")
print("probabilities at (b=0,t=0):", probs[0, 0].round(3), "  sum =", probs[0, 0].sum().round(3))

> 🧒 **In plain words:**
> The decoder hands us 8 numbers for each of 4 positions in 2 examples. The matrix `W_out` multiplies them to give 13 scores per position, one for every word in our 13-word dictionary. These raw scores are the **logits**.
> The shape `(B, T, V)` means: B examples, each with T positions, each position holding V scores, one for every word in the dictionary (here 2, 4 and 13).
> `softmax` then turns the 13 scores into percentages.
> **Notice:** the probabilities for the first position are all small-ish (the biggest is about 0.23) and add up to 1.0. The model is untrained, so it has no strong opinion.

*(The paper also ties `W_out` to the embedding table transposed, saving `V·d` parameters.)*

## 3. Cross-entropy loss
For each position, take the probability the model gave to the **correct** token and compute `−log p`:

- gave the right token probability 1.0 → loss 0
- probability 0.01 → loss ≈ 4.6

Average over all (non-padding) positions.

> 🧒 **Think of it like...** a quiz where you must give a percentage confidence for each answer. If you said 99% for the right answer you lose almost nothing; if you said 1% for the right answer you lose lots. `-log p` is that penalty, and training tries to shrink the average penalty.

In [ ]:
targets = rng.integers(0, V, size=(B, T))
print("targets:\n", targets)

p_correct = np.take_along_axis(probs, targets[..., None], axis=-1)[..., 0]   # (B,T)
loss_per_pos = -np.log(p_correct)
print("\nP(correct):\n", p_correct.round(3))
print("loss per position:\n", loss_per_pos.round(2))
print("\nmean loss:", loss_per_pos.mean().round(3))

> 🧒 **In plain words:**
> We pick a random "correct answer" for every position (`targets`). Then we ask: how much probability did the model give to the correct word?
> The tricky line is `np.take_along_axis`. For each position we have 13 probabilities, and this line picks out just one of them: the one at the position of the correct word. Think of it as looking up one entry in each row of a table. `[..., 0]` just removes a spare dimension.
> `-np.log(p)` turns probability into a penalty: probability near 1 gives near 0 penalty, small probability gives a big penalty.
> **Notice:** the probabilities are all small (0.013 to 0.088) so every position gets a penalty of 2.4 to 4.3. The smallest probability (0.013) got the biggest penalty (4.32). The average is 3.33.
> **Try it:** set a target to the word with the highest probability and watch its penalty shrink.

### Baseline: what does "knows nothing" cost?
A uniform guess over `V` tokens gives loss `ln V`. A freshly initialized model should start near this; if not, something is off.

In [ ]:
print("ln(V) =", np.log(V).round(3), "  (compare with the mean loss above)")
print("perplexity of uniform guess =", np.exp(np.log(V)).round(1), "= V: 'as unsure as choosing among V options'")

> 🧒 **In plain words:**
> `np.log(V)` is the penalty you would get by guessing completely evenly among all 13 words. `np.exp(np.log(V))` undoes the log and gives back 13: the perplexity, meaning "as unsure as choosing among 13 options".
> **Notice:** the guess-evenly penalty is 2.565, lower than our mean loss of 3.33. Our random model is actually a bit worse than just shrugging, because it is unsure but also leaning towards some wrong words. A real freshly built model should start close to 2.565.

### Ignoring padding in the loss
Multiply by a mask so `<PAD>` positions don't count:

In [ ]:
mask = np.array([[1, 1, 1, 1],
                 [1, 1, 0, 0]], dtype=float)      # example 1 has only 2 real tokens
masked_loss = (loss_per_pos * mask).sum() / mask.sum()
print("masked mean loss:", masked_loss.round(3), "(divides by", int(mask.sum()), "real tokens, not", mask.size, ")")

> 🧒 **In plain words:**
> Sentences have different lengths, so shorter ones get filler `<PAD>` words at the end. We should not mark the model on filler!
> The `mask` is a table of 1s (real word) and 0s (filler). Multiplying the penalties by it turns the filler penalties into 0. Then we add up and divide by the number of real words (`mask.sum()`), not by all 8 slots.
> **Notice:** it divides by 6, not 8, and the result (3.379) is the average of only the six real penalties.
> **Try it:** change a 0 to a 1 in the mask and see the divisor change.

### What the loss curve looks like when learning
A toy: gradient descent on logits for a fixed target pushes `P(correct) → 1` and loss → 0.

In [ ]:
z = np.zeros(V); target = 5; lr = 0.5; losses = []
for step in range(40):
    p = softmax(z)
    losses.append(-np.log(p[target]))
    grad = p.copy(); grad[target] -= 1          # d(loss)/d(logits) = p - one_hot
    z -= lr * grad
plt.figure(figsize=(5, 3)); plt.plot(losses); plt.axhline(np.log(V), ls="--", c="gray")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("starts at ln V, falls toward 0"); plt.show()

> 🧒 **In plain words:**
> Here we watch learning in slow motion. We start with all 13 scores equal to 0 (no opinion) and target word number 5. The loop repeats 40 times: work out the percentages, record the penalty, then nudge the scores. `grad` is the direction of "more wrong", and the line `grad[target] -= 1` makes the correct word's score go up and all the others go down. `z -= lr * grad` takes a small step (size `lr`, the learning rate).
> **Notice in the graph:** it starts at ln 13, about 2.56 (the even-guess penalty), and falls steadily towards 0 as the model becomes sure of word 5. The dashed line marks that starting level.
> **Try it:** change `lr = 0.5` to `0.05` and the curve falls much more slowly.

## 4. Training vs. inference
| | training | inference |
|---|---|---|
| decoder input | whole correct answer, shifted (teacher forcing) | tokens generated *so far* |
| positions computed | **all in parallel** (causal mask prevents cheating) | **one new token per step** |
| output used | loss at every position | only the last position's distribution |

> 🧒 **Think of it like...** training is a teacher marking a whole page at once, with the correct previous words already filled in, so every position is checked at the same time. Inference is a student writing the answer live, one word at a time, with no answer sheet to peek at.

## 5. Autoregressive generation
Start with `<SOS>`; repeatedly: run the decoder on everything produced so far → distribution over next token → **pick one** → append → stop at `<EOS>`.

To keep this notebook standalone we use a hand-made "model" (a table of next-token probabilities that sorts nothing — it just continues a counting pattern) in place of a trained network, so you can focus on the **loop**.

In [ ]:
SOS, EOS = 11, 12
Vg = 13

def toy_next_token_probs(prefix):
    """Stand-in for a trained decoder: favours (last token + 1), ends after 5 digits."""
    p = np.full(Vg, 0.01)
    if len(prefix) > 5:
        p[EOS] = 1.0
    else:
        last = 0 if prefix[-1] == SOS else prefix[-1]
        p[(last + 1) % 10] = 1.0
        p[(last + 2) % 10] = 0.3
    return p / p.sum()

def greedy_decode(max_len=10):
    seq = [SOS]
    while len(seq) < max_len:
        probs = toy_next_token_probs(seq)
        nxt = int(np.argmax(probs))           # greedy: take the most likely
        seq.append(nxt)
        print(f"step {len(seq)-1}: prefix={seq[:-1]!s:28} -> next={nxt} (p={probs[nxt]:.2f})")
        if nxt == EOS:
            break
    return seq

print(greedy_decode())

> 🧒 **In plain words:**
> This is the writing loop. `toy_next_token_probs` is a pretend model: it likes "the last number plus 1", slightly likes "plus 2", and gives everything else a tiny chance. After 5 digits it wants the end marker `<EOS>` (here number 12). `SOS` (11) is the start marker.
> In `greedy_decode`: start with `[SOS]`; ask the model for percentages; `np.argmax` picks the position of the biggest one (the favourite); add it to the list; print; stop if it was `<EOS>`. Each new word depends on the ones before it, that is **autoregressive**.
> **Notice:** the sequence counts 1, 2, 3, 4, 5 and then ends with 12. The favourite has probability 0.71 each time, then 0.89 for the end marker.
> **Try it:** change `max_len=10` to `4` and generation stops early, before the end marker.

## 6. Sampling and temperature
Greedy is deterministic. To get variety, **sample** from the distribution. **Temperature** `τ` reshapes it: `softmax(logits / τ)`.

- `τ < 1` → sharper (more greedy), `τ > 1` → flatter (more random).

In [ ]:
logits = np.array([2.0, 1.0, 0.5, 0.0, -1.0])
fig, ax = plt.subplots(figsize=(6, 3))
for tau, off in [(0.3, -0.25), (1.0, 0.0), (3.0, 0.25)]:
    ax.bar(np.arange(5) + off, softmax(logits / tau), width=0.25, label=f"τ={tau}")
ax.set_xlabel("token"); ax.set_ylabel("probability"); ax.legend(); plt.show()

rng = np.random.default_rng(0)
for tau in [0.3, 1.0, 3.0]:
    draws = rng.choice(5, size=1000, p=softmax(logits / tau))
    print(f"τ={tau}: share of token 0 = {np.mean(draws == 0):.2f}")

> 🧒 **In plain words:**
> Five made-up scores are turned into percentages three times, dividing the scores by the temperature `tau` first. Low `tau` (0.3) exaggerates the differences so the top word dominates; high `tau` (3.0) shrinks them so everything gets a similar chance.
> The loop then "rolls the dice" 1000 times with `rng.choice`, using those percentages, and counts how often token 0 came up.
> **Notice:** in the bar chart the tau=0.3 bars have one tall bar; the tau=3.0 bars are nearly level. The printed shares of token 0 fall from 0.96 to 0.59 to 0.30.
> **Try it:** set `tau` to 0.05 and token 0 should win almost every time, like greedy.

## 7. Putting the whole picture together
```
source ids (B,T_enc) ─► embed*√d + PE ─► [Encoder × N] ─► memory (B,T_enc,d)
                                                              │ (K,V)
target ids (B,T_dec) ─► embed*√d + PE ─► [Decoder × N] ◄──────┘
                                              │
                                         Linear (d→V) ─► logits (B,T_dec,V) ─► softmax ─► loss / next token
```
You've now met every shape and every piece. To see them wired together in real code — with backprop written out by hand — open:

- `numpy-transformer/src/numpy_transformer/layers.py`
- `docs/00_overview.md` … `docs/09_inference.md`

## Recap
- Decoder = causal self-attention + cross-attention to the encoder + FFN.
- Output logits `(B,T,V)` → softmax → cross-entropy against the next token.
- Training is parallel with teacher forcing; inference is a sequential loop.